In [9]:

!pip -q install ipywidgets
from google.colab import output
output.enable_custom_widget_manager()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output


def clamp(x, lo=0.0, hi=200.0):
    return np.minimum(np.maximum(x, lo), hi)

def monod(x, k):
    x = np.maximum(x, 0.0)
    return x / (k + x + 1e-12)

def light_intensity(light_mode):
    return {"Purple": 2.2, "White": 1.0, "Green": 0.25}[light_mode]

def filter_params(filter_mode):
    return {
        "High":   {"k_nh3": 1.6, "k_no2": 0.9, "stress": 0.015},
        "Medium": {"k_nh3": 0.8, "k_no2": 0.4, "stress": 0.004},
        "Low":    {"k_nh3": 0.25,"k_no2": 0.12,"stress": 0.001},
    }[filter_mode]

IDX = {"O2":0, "CO2":1, "NH3":2, "NO2":3, "NO3":4, "A":5, "F":6}

# -----------------------------
# Parameters (tuned for visible dynamics)
# -----------------------------
P = {
    "p_max": 1.8, "K_I": 0.9, "K_CO2": 25.0, "K_N": 22.0,
    "y_A": 0.10, "a_resp": 0.22, "a_death": 0.03, "K_A": 90.0,

    "f_resp": 0.55, "w_nh3": 0.65, "f_growth": 0.006,

    "k_n1": 0.30, "k_n2": 0.22, "K_O2_nit": 35.0, "k_nit_o2": 0.18,

    "k_ex_O2": 0.04, "k_ex_CO2": 0.05, "O2_sat": 105.0, "CO2_sat": 40.0,

    "k_photo_co2": 0.90,
    "u_N": 0.22,

    "NH3_safe": 28.0, "NH3_scale": 30.0,
    "O2_safe": 55.0, "O2_scale": 30.0,
    "m_base": 0.002, "m_nh3": 0.045, "m_o2": 0.030,
}

# -----------------------------
# ODE derivatives
# -----------------------------
def derivatives(t, y, p, controls, rng):
    O2, CO2, NH3, NO2, NO3, A, F = y

    I = light_intensity(controls["light"])
    fp = filter_params(controls["filter"])
    kf_nh3, kf_no2 = fp["k_nh3"], fp["k_no2"]

    noise = float(controls.get("noise", 0.0))
    shock = float(controls.get("shock", 0.0))

    eps_photo = eps_waste = eps_filter = 1.0
    if noise > 0:
        eps_photo  = np.clip(rng.normal(1.0, noise), 0.2, 2.5)
        eps_waste  = np.clip(rng.normal(1.0, noise), 0.2, 2.5)
        eps_filter = np.clip(rng.normal(1.0, noise), 0.2, 2.5)

    if shock > 0:
        if rng.random() < 0.03 * shock: eps_waste *= 2.2
        if rng.random() < 0.03 * shock: eps_filter *= 0.4

    photo = (p["p_max"] * A
             * monod(I, p["K_I"])
             * monod(CO2, p["K_CO2"])
             * monod((NO3 + NH3), p["K_N"])
             * eps_photo)

    alg_resp  = p["a_resp"] * A
    fish_resp = p["f_resp"] * F

    dA = p["y_A"] * photo - p["a_death"] * A * (1.0 + A/p["K_A"])

    waste = p["w_nh3"] * F * eps_waste

    nitr1 = p["k_n1"] * NH3 * monod(O2, p["K_O2_nit"])
    nitr2 = p["k_n2"] * NO2 * monod(O2, p["K_O2_nit"])

    filt_nh3 = (kf_nh3 * NH3) * eps_filter
    filt_no2 = (kf_no2 * NO2) * eps_filter

    exch_O2  = p["k_ex_O2"]  * (p["O2_sat"]  - O2)
    exch_CO2 = p["k_ex_CO2"] * (p["CO2_sat"] - CO2)

    tox_nh3 = np.maximum(0.0, NH3 - p["NH3_safe"]) / (p["NH3_scale"] + 1e-12)
    hypox   = np.maximum(0.0, p["O2_safe"] - O2)   / (p["O2_scale"] + 1e-12)
    hazard  = p["m_base"] + p["m_nh3"]*(tox_nh3**2) + p["m_o2"]*(hypox**2) + fp["stress"]
    dF = p["f_growth"]*F - hazard*F

    N_uptake = p["u_N"] * photo
    frac_nh3 = (NH3 / (NH3 + NO3 + 1e-12))
    uptake_nh3 = N_uptake * (0.7 + 0.3*frac_nh3)
    uptake_no3 = np.maximum(0.0, N_uptake - uptake_nh3)

    dO2  = + photo - alg_resp - fish_resp + exch_O2 - p["k_nit_o2"]*(nitr1 + nitr2)
    dCO2 = - p["k_photo_co2"]*photo + alg_resp + fish_resp + exch_CO2

    dNH3 = + waste - uptake_nh3 - nitr1 - filt_nh3
    dNO2 = + nitr1 - nitr2 - filt_no2
    dNO3 = + nitr2 - uptake_no3

    return np.array([dO2, dCO2, dNH3, dNO2, dNO3, dA, dF], dtype=float)

# -----------------------------
# Integrators
# -----------------------------
def step_euler(t, y, step_days, p, controls, rng):
    return y + step_days * derivatives(t, y, p, controls, rng)

def step_rk4(t, y, step_days, p, controls, rng):
    h = step_days
    k1 = derivatives(t, y, p, controls, rng)
    k2 = derivatives(t + 0.5*h, y + 0.5*h*k1, p, controls, rng)
    k3 = derivatives(t + 0.5*h, y + 0.5*h*k2, p, controls, rng)
    k4 = derivatives(t + h, y + h*k3, p, controls, rng)
    return y + (h/6.0) * (k1 + 2*k2 + 2*k3 + k4)

def get_stepper(method):
    return {"Euler": step_euler, "RK4": step_rk4}[method]

# -----------------------------
# Simulation (records every step + daily snapshot)
# -----------------------------
def simulate(days, step_days, method, fish0, algae0, light, filter_mode, noise, shock, seed):
    stepper = get_stepper(method)
    rng = np.random.default_rng(int(seed))
    controls = {"light": light, "filter": filter_mode, "noise": float(noise), "shock": float(shock)}

    y = np.array([100.0, 50.0, 10.0, 0.0, 10.0, float(algae0), float(fish0)], dtype=float)

    T = float(days)
    n_steps = int(np.ceil(T / step_days))
    t = 0.0

    rows = []
    for _ in range(n_steps + 1):
        rows.append({
            "Time": t,
            "O2": y[IDX["O2"]],
            "NH3": y[IDX["NH3"]],
            "NO2": y[IDX["NO2"]],
            "NO3": y[IDX["NO3"]],
            "Algae": y[IDX["A"]],
            "Fish": y[IDX["F"]],
        })

        y = stepper(t, y, step_days, P, controls, rng)

        y[0:5] = clamp(y[0:5], 0.0, 200.0)
        y[IDX["A"]] = max(0.0, y[IDX["A"]])
        y[IDX["F"]] = max(0.0, y[IDX["F"]])

        t = min(T, t + step_days)
        if t >= T - 1e-12:
            break

    df = pd.DataFrame(rows)

    # FIX: build daily snapshot WITHOUT duplicate "Day"
    df_day = df.copy()
    df_day["Day"] = np.floor(df_day["Time"] + 1e-9).astype(int)
    df_day = df_day.sort_values("Time").drop_duplicates("Day", keep="last")
    df_day = df_day[["Day","O2","NH3","NO2","NO3","Algae","Fish"]].reset_index(drop=True)
    return df, df_day

def simulate_mc(runs, **kwargs):
    runs = int(runs)
    base_seed = int(kwargs["seed"])
    out = []
    for r in range(runs):
        df, df_day = simulate(seed=base_seed + r, **{k:v for k,v in kwargs.items() if k!="seed"})
        df_day["run"] = r
        out.append(df_day)
    return pd.concat(out, ignore_index=True)

# -----------------------------
# Dashboard
# -----------------------------
def dashboard_html(df_day):
    last = df_day.iloc[-1]
    min_o2 = float(df_day["O2"].min())
    max_nh3 = float(df_day["NH3"].max())
    max_no2 = float(df_day["NO2"].max())

    alerts = []
    if min_o2 < 55: alerts.append("O2 LOW")
    if max_nh3 > 28: alerts.append("NH3 HIGH")
    if max_no2 > 5:  alerts.append("NO2 HIGH")
    alert_line = "✅ OK" if not alerts else "⚠️ " + " | ".join(alerts)

    d = last.to_dict()
    day_last = int(d["Day"])

    card = lambda title, big, small: f"""
      <div class="card">
        <div class="muted">{title}</div>
        <div class="big">{big}</div>
        <div class="muted2">{small}</div>
      </div>
    """

    return f"""
    <style>
      .panel {{
        background:#1e1f22; border:1px solid #2b2d31; border-radius:14px;
        padding:14px; color:#e6e6e6; font-family:Arial;
      }}
      .title {{ font-size:16px; font-weight:900; margin:0 0 10px 0; }}
      .grid {{
        display:grid; grid-template-columns: repeat(2, minmax(220px, 1fr));
        gap:10px;
      }}
      .card {{
        background:#232427; border:1px solid #2f3136; border-radius:14px;
        padding:12px;
      }}
      .muted {{ opacity:.75; font-size:12px; margin-bottom:6px; }}
      .muted2 {{ opacity:.75; font-size:12px; margin-top:8px; }}
      .big {{ font-size:20px; font-weight:900; }}
      .alert {{
        margin-top:10px; background:#232427; border:1px solid #2f3136;
        border-radius:14px; padding:10px 12px; font-weight:900;
      }}
    </style>

    <div class="panel">
      <div class="title">Easy Dashboard</div>
      <div class="grid">
        {card("Last Day", f"Day {day_last}", f"Fish / Algae: {d['Fish']:.2f} / {d['Algae']:.2f}")}
        {card("O2", f"{d['O2']:.1f}", f"Min: {min_o2:.1f}")}
        {card("NH3", f"{d['NH3']:.1f}", f"Max: {max_nh3:.1f}")}
        {card("NO2", f"{d['NO2']:.2f}", f"Max: {max_no2:.2f}")}
      </div>
      <div class="alert">{alert_line}</div>
    </div>
    """

# -----------------------------
# Plotting
# -----------------------------
def plot_single(df, title):
    plt.figure(figsize=(10.5,4.5))
    plt.plot(df["Time"], df["O2"],  label="O2")
    plt.plot(df["Time"], df["NH3"], label="NH3")
    plt.plot(df["Time"], df["NO2"], label="NO2")
    plt.plot(df["Time"], df["NO3"], label="NO3")
    plt.xlabel("Time (days)"); plt.ylabel("Level")
    plt.grid(alpha=0.25); plt.legend()
    plt.title(title + " | Chemicals (smooth)")
    plt.show()

    plt.figure(figsize=(10.5,4.0))
    plt.plot(df["Time"], df["Fish"],  label="Fish")
    plt.plot(df["Time"], df["Algae"], label="Algae")
    plt.xlabel("Time (days)"); plt.ylabel("Population / Biomass")
    plt.grid(alpha=0.25); plt.legend()
    plt.title(title + " | Populations (smooth)")
    plt.show()

def plot_mc(df_all, title):
    g = df_all.groupby("Day")

    def band(col):
        mean = g[col].mean()
        lo = g[col].quantile(0.10)
        hi = g[col].quantile(0.90)
        return mean.index.values, mean.values, lo.values, hi.values

    plt.figure(figsize=(10.5,4.5))
    for col in ["O2","NH3","NO2","NO3"]:
        x, m, lo, hi = band(col)
        plt.plot(x, m, label=f"{col} mean")
        plt.fill_between(x, lo, hi, alpha=0.18)
    plt.xlabel("Day"); plt.ylabel("Level")
    plt.grid(alpha=0.25); plt.legend()
    plt.title(title + " | Monte Carlo (10–90% band)")
    plt.show()

    plt.figure(figsize=(10.5,4.0))
    x, m, lo, hi = band("Fish")
    plt.plot(x, m, label="Fish mean")
    plt.fill_between(x, lo, hi, alpha=0.18)
    plt.xlabel("Day"); plt.ylabel("Fish")
    plt.grid(alpha=0.25); plt.legend()
    plt.title("Fish | Monte Carlo band")
    plt.show()

# -----------------------------
# UI
# -----------------------------
controls = {
    "fish0": widgets.IntSlider(min=1, max=100, value=15, description="Fish0"),
    "algae0": widgets.IntSlider(min=0, max=150, value=20, description="Algae0"),
    "light": widgets.Dropdown(options=["Purple","White","Green"], value="Purple", description="Light"),
    "filter": widgets.Dropdown(options=["High","Medium","Low"], value="Medium", description="Filter"),
    "days": widgets.IntSlider(min=1, max=60, value=20, description="Days"),
    "step_days": widgets.FloatSlider(min=0.10, max=1.0, step=0.10, value=0.20, description="Step (days)"),
    "method": widgets.Dropdown(options=["Euler","RK4"], value="RK4", description="Method"),
    "runs": widgets.IntSlider(min=1, max=120, value=1, description="Runs"),
    "noise": widgets.FloatSlider(min=0.0, max=0.8, step=0.05, value=0.15, description="Noise"),
    "shock": widgets.FloatSlider(min=0.0, max=1.0, step=0.10, value=0.4, description="Shock"),
    "seed": widgets.IntSlider(min=0, max=20, value=0, description="Seed"),
}

btn_rand_seed = widgets.Button(description="Randomize seed", button_style="")
dash_out = widgets.Output()
plot_out = widgets.Output()

left_box = widgets.VBox(list(controls.values()) + [btn_rand_seed], layout=widgets.Layout(width="460px"))
right_box = widgets.VBox([dash_out, plot_out], layout=widgets.Layout(width="820px"))
ui = widgets.HBox([left_box, right_box])

def render(*_):
    with dash_out: clear_output(wait=True)
    with plot_out: clear_output(wait=True)

    fish0 = int(controls["fish0"].value)
    algae0 = int(controls["algae0"].value)
    light = controls["light"].value
    filt = controls["filter"].value
    days = int(controls["days"].value)
    step_days = float(controls["step_days"].value)
    method = controls["method"].value
    runs = int(controls["runs"].value)
    noise = float(controls["noise"].value)
    shock = float(controls["shock"].value)
    seed = int(controls["seed"].value)

    title = f"Method={method} | Step={step_days} day | Light={light} | Filter={filt} | runs={runs} | noise={noise} | shock={shock}"

    if runs <= 1:
        df, df_day = simulate(days=days, step_days=step_days, method=method, fish0=fish0, algae0=algae0,
                              light=light, filter_mode=filt, noise=noise, shock=shock, seed=seed)
        with dash_out: display(HTML(dashboard_html(df_day)))
        with plot_out: plot_single(df, title)
    else:
        df_all = simulate_mc(runs=runs, days=days, step_days=step_days, method=method, fish0=fish0, algae0=algae0,
                             light=light, filter_mode=filt, noise=noise, shock=shock, seed=seed)
        df0 = df_all[df_all["run"] == 0].drop(columns=["run"]).reset_index(drop=True)
        with dash_out: display(HTML(dashboard_html(df0)))
        with plot_out: plot_mc(df_all, title)

def randomize_seed(_):
    controls["seed"].value = int(np.random.default_rng().integers(0, 21))

btn_rand_seed.on_click(randomize_seed)

for w in controls.values():
    w.observe(render, names="value")

display(ui)
render()
